# Traffic YOLO — entraînement

7 classes : `Car`, `Number Plate`, `Blur Number Plate`, `Two Wheeler`, `Auto`, `Bus`, `Truck`.
Sortie : **`best.onnx`** à déposer dans `web/models/` du front, et `best.pt`.

*Exécution → Modifier le type d'exécution → **GPU T4*** avant de commencer.

In [ ]:
%pip install -q ultralytics onnx onnxslim onnxruntime
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 1. Dataset

Envoyez `archive.zip`, ou glissez-le dans le panneau *Fichiers* et sautez cette cellule.
L'archive contient déjà `images/{train,val}` et `labels/{train,val}` : on garde ce découpage.

In [ ]:
import zipfile, yaml
from pathlib import Path

if not Path('archive.zip').exists():
    from google.colab import files
    files.upload()

ROOT = Path('/content/data')
if not ROOT.exists():
    with zipfile.ZipFile('archive.zip') as z:
        z.extractall(ROOT)

BASE = next(p.parent.parent for p in ROOT.rglob('images/train') if p.is_dir())
CLASS_NAMES = ['Car', 'Number Plate', 'Blur Number Plate', 'Two Wheeler', 'Auto', 'Bus', 'Truck']

DATA_YAML = Path('/content/data.yaml')
DATA_YAML.write_text(yaml.safe_dump({
    'path': str(BASE), 'train': 'images/train', 'val': 'images/val',
    'nc': len(CLASS_NAMES), 'names': dict(enumerate(CLASS_NAMES)),
}, sort_keys=False))

for split in ('train', 'val'):
    n = len(list((BASE / 'labels' / split).glob('*.txt')))
    print(f'{split}: {n} images annotees')

## 2. Entraînement

`yolo11s` : ~20 Mo en ONNX, bon compromis pour une inférence navigateur.
`yolo11n` si le front doit tourner sur mobile, `yolo11m` pour la précision.
Environ 35 min sur T4. `patience=25` arrête tôt si le mAP stagne.

Si `Number Plate` plafonne, augmentez `IMGSZ` à 960 plutôt que `EPOCHS` :
les petits objets ne sont vus que par la carte de détection la plus fine.

In [ ]:
from ultralytics import YOLO

IMGSZ, EPOCHS = 640, 100

results = YOLO('yolo11s.pt').train(
    data=str(DATA_YAML), epochs=EPOCHS, imgsz=IMGSZ, batch=16,
    device=0, workers=2, patience=25, seed=42,
    project='/content/runs', name='traffic-yolo', exist_ok=True, plots=True,
    # augmentations adaptees a des scenes routieres : pas de flip vertical
    fliplr=0.5, flipud=0.0, degrees=5.0, translate=0.1, scale=0.5, shear=2.0,
    mosaic=1.0, close_mosaic=10,
)
BEST = Path(results.save_dir) / 'weights' / 'best.pt'

## 3. Résultats

In [ ]:
from IPython.display import Image, display

for name in ('results.png', 'confusion_matrix_normalized.png', 'val_batch0_pred.jpg'):
    p = Path(results.save_dir) / name
    if p.exists():
        print(name); display(Image(filename=str(p), width=900))

m = YOLO(BEST).val(data=str(DATA_YAML), device=0, split='val')
order = {int(c): i for i, c in enumerate(m.box.ap_class_index)}
print(f"\n{'classe':<20}{'P':>8}{'R':>8}{'mAP50':>10}{'mAP50-95':>10}")
for i, name in enumerate(CLASS_NAMES):
    if i in order:
        p, r, ap50, ap = m.box.class_result(order[i])
        print(f'{name:<20}{p:>8.3f}{r:>8.3f}{ap50:>10.3f}{ap:>10.3f}')
print(f'\nmAP50 {m.box.map50:.4f}   mAP50-95 {m.box.map:.4f}')

## 4. Export ONNX

`nms=False` et `dynamic=False` : onnxruntime-web ne couvre que partiellement les
opérateurs de NMS, le front fait donc lui-même le décodage et le NMS.
`opset=12` pour la compatibilité avec le backend WASM.

La sortie doit être `[1, 11, 8400]` — soit 4 coordonnées + 7 classes, sur
80² + 40² + 20² positions candidates.

In [ ]:
import onnxruntime as ort

onnx_path = YOLO(BEST).export(format='onnx', imgsz=IMGSZ, opset=12,
                              simplify=True, dynamic=False, nms=False, half=False)

s = ort.InferenceSession(str(onnx_path), providers=['CPUExecutionProvider'])
print('entree', s.get_inputs()[0].shape, '-> sortie', s.get_outputs()[0].shape)
print(f'{Path(onnx_path).stat().st_size / 1e6:.1f} Mo')

## 5. Téléchargement

Déposez `best.onnx` dans `web/models/` du projet front, et `best.pt` à la racine
(`evaluate.py` s'en sert pour mesurer le mAP par classe).

In [ ]:
from google.colab import files

files.download(str(onnx_path))
files.download(str(BEST))